In [1]:
!pip install requests beautifulsoup4 pandas tqdm -q

In [2]:
import requests
import re
from bs4 import BeautifulSoup
import json
import time
import random
from tqdm import tqdm
import pandas as pd
from urllib.parse import urljoin

Configuración


In [3]:
# Voy a probar con recetas.com por que es de las pocas con un formato limpio y consistente (sin ser un artículo)
BASE_URL = "https://www.recetas.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9"
}

session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url):
    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'html.parser')
    except Exception as e:
        print(f"Error al acceder a {url}: {e}")
        return None

Extracción de URL de las recetas

In [4]:
#La página cuenta con una sección /recetas con enlaces a cada receta.
#La iremos recoriendo para obtener sus enlaces.
def get_recipe_links(page=1):
    if page == 1:
        url = f"{BASE_URL}/recetas/"
    else:
        url = f"{BASE_URL}/recetas/{page}/"

    soup = get_soup(url)
    if not soup:
        print(f"No se pudo obtener la página {page}")
        return []

    links = []

    for a in soup.find_all('a', href=True):
        href = a['href'].strip()

        if (href.endswith('.html') and
            len(href) > 20 and
            not any(folder in href for folder in ['/recetas/', '/videos/', '/chefs/', '/reportajes/', '/especiales/', '/utilidades/'])):

            full_url = urljoin(BASE_URL, href)
            if full_url not in links:
                links.append(full_url)

    recipe_links = [url for url in links if url.count('-') >= 3]

    return recipe_links

Limpieza de ingredientes

In [5]:
def limpiar_ingredientes(ingredientes_raw):
    if not isinstance(ingredientes_raw, str) or not ingredientes_raw.strip():
        return []

    text = re.sub(r'\s+', ' ', ingredientes_raw)
    text = re.sub(r'(\d+)\s*gr\.', r'\1 gr.', text, flags=re.I)
    text = re.sub(r'(\d+)\s*ml\.', r'\1 ml.', text, flags=re.I)
    text = re.sub(r'(\d+)\s*cdas?\.?', r'\1 cdas.', text, flags=re.I)
    text = re.sub(r'(\d+)\s*cdtas?\.?', r'\1 cdta.', text, flags=re.I)

    lines = [line.strip() for line in text.split('\n') if line.strip()]
    final = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if any(phrase in line.lower() for phrase in [
            "para la base", "para el relleno", "para cubrir", "para la masa",
            "para la crema", "para el", "ingredientes", "cómo hacer", "preparación"]):
            i += 1
            continue

        if i + 1 < len(lines):
            next_line = lines[i+1].strip()
            if (len(next_line) < 30 or
                re.search(r'^(de|con|sin|al|a la|para|picado|c/n|cucharad)', next_line.lower()) or
                any(u in next_line.lower() for u in ['gr.', 'ml.', 'cdas', 'cdta', 'cucharad'])):
                line = (line + " " + next_line).strip()
                i += 1

        line = re.sub(r'\s+', ' ', line).strip()

        if line and len(line) > 3 and not line.endswith(':'):
            final.append(line)

        i += 1

    return final

Limpieza de instrucciones

In [6]:
def limpiar_instrucciones(instrucciones_raw):
    if not isinstance(instrucciones_raw, str) or not instrucciones_raw.strip():
        return []

    text = re.sub(r'(\w)([A-ZÁÉÍÓÚÑ])', r'\1. \2', instrucciones_raw)
    text = re.sub(r'\s+', ' ', text)

    sentences = re.split(r'[.\n]', text)
    final = []

    for sent in sentences:
        sent = sent.strip()
        if len(sent) > 20:
            final.append(sent.rstrip('.') + '.')

    return final

Extracción de receta individual

In [7]:
#Metodo para scrapear url de receta específica.
def scrape_recipe(url):
    soup = get_soup(url)
    if not soup:
        return None

    recipe = {
        "url": url,
        "titulo": None,
        "ingredientes": [],
        "instrucciones": [],
        "tiempo_total": None,
        "tiempo_preparacion": None,
        "tiempo_coccion": None,
        "dificultad": None,
        "porciones": None,
        "autor": None,
        "fecha": None
    }

    try:
        # === JSON-LD ===
        json_ld = soup.find('script', type='application/ld+json')
        if json_ld:
            try:
                data = json.loads(json_ld.string)
                if isinstance(data, dict) and data.get('@type') == 'Recipe':
                    recipe["titulo"] = data.get("name")
                    recipe["autor"] = data.get("author", {}).get("name") if isinstance(data.get("author"), dict) else data.get("author")
                    recipe["porciones"] = data.get("recipeYield")
            except:
                pass

        # Título fallback
        if not recipe["titulo"]:
            h1 = soup.find('h1')
            if h1:
                recipe["titulo"] = h1.get_text(strip=True)

        # Autor
        author_li = soup.find('li', class_='recipe_author')
        if author_li:
            a_tag = author_li.find('a')
            if a_tag:
                recipe["autor"] = a_tag.get_text(strip=True)

        # Datos de receta (dificultad, porciones, tiempo)
        recipe_data = soup.find('div', class_='recipe_data')
        if recipe_data:
            diff = recipe_data.find('div', class_='difficulty')
            if diff:
                recipe["dificultad"] = diff.get_text(strip=True)

            servings = recipe_data.find('div', class_='servings')
            if servings:
                recipe["porciones"] = servings.get_text(strip=True)

            time_div = recipe_data.find('div', class_='time')
            if time_div:
                recipe["tiempo_total"] = time_div.get_text(strip=True)

        # Procesar tiempos
        tiempo_texto = recipe.get("tiempo_total") or ""
        if tiempo_texto:
            numbers = re.findall(r'(\d+)', tiempo_texto)
            if numbers:
                recipe["tiempo_total"] = f"{numbers[0]} minutos"
                if len(numbers) > 1:
                    recipe["tiempo_preparacion"] = f"{numbers[1]} minutos"
                if len(numbers) > 2:
                    recipe["tiempo_coccion"] = f"{numbers[2]} minutos"

        # === INGREDIENTES (con limpiador potente) ===
        ingredients_div = soup.find('div', class_='ingredients_info')
        if ingredients_div:
            raw = ingredients_div.get_text(separator="\n", strip=True)
            recipe["ingredientes"] = limpiar_ingredientes(raw)

        # === INSTRUCCIONES (con limpiador potente) ===
        prep_div = soup.find('div', class_='preparation_info')
        if prep_div:
            raw_prep = prep_div.get_text(separator="\n", strip=True)
            recipe["instrucciones"] = limpiar_instrucciones(raw_prep)

    except Exception as e:
        print(f"Error en {url}: {e}")

    return recipe

Scrapear varias páginas de sección recetas

In [8]:
#Metodo para scrapear varias paginaciones de /recetas
def scrape_multiple_pages(max_pages=5, delay=(3, 6)):
    all_recipes = []
    all_links = []

    print(f"Extrayendo enlaces de las primeras {max_pages} páginas...")
    for page in tqdm(range(1, max_pages + 1)):
        links = get_recipe_links(page)
        all_links.extend(links)
        time.sleep(random.uniform(1.5, 3))

    all_links = list(dict.fromkeys(all_links))
    print(f"Se encontraron {len(all_links)} URLs únicas de recetas.")

    print("Scraping de recetas individuales...")
    for url in tqdm(all_links):
        try:
            recipe = scrape_recipe(url)
            if recipe and recipe.get("titulo"):
                all_recipes.append(recipe)
            else:
                print(f"  → Saltada (sin título): {url}")
        except Exception as e:
            print(f"  → Error en {url}: {e}")

        time.sleep(random.uniform(*delay))

    print(f"\nScraping completado. Se obtuvieron {len(all_recipes)} recetas válidas.")
    return all_recipes

Ejeccución final de los metodos

In [9]:
#recipes = scrape_multiple_pages(max_pages=3, delay=(3, 5))
#
#print(f"\nTotal de recetas extraídas correctamente: {len(recipes)}")
#
#if recipes:
#   print("\nEjemplo de la primera receta:")
#    print(json.dumps(recipes[0], indent=2, ensure_ascii=False))
#
# Guardar
# df = pd.DataFrame(recipes)
# df.to_json("recetas_com.json", orient="records", force_ascii=False, indent=2)
# df.to_csv("recetas_com.csv", index=False)
# print("Archivos guardados correctamente.")

# ====================== EJECUCIÓN FINAL ======================
print("Iniciando scraping masivo...")

recipes = scrape_multiple_pages(max_pages=8, delay=(3, 6))   # puedes subir a 10-15

print(f"\nSe extrajeron {len(recipes)} recetas.")

# Convertir a DataFrame
df = pd.DataFrame(recipes)

# Convertir listas a strings para Excel
df['ingredientes'] = df['ingredientes'].apply(lambda x: "\n".join(x) if isinstance(x, list) else x)
df['instrucciones'] = df['instrucciones'].apply(lambda x: "\n".join(x) if isinstance(x, list) else x)

# Guardar archivos
df.to_excel("recetas_extraidas.xlsx", index=False)
df.to_json("recetas_extraidas.json", orient="records", force_ascii=False, indent=2)

print("\n✅ Archivos guardados correctamente:")
print("- recetas_extraidas.xlsx")
print("- recetas_extraidas.json")

# Mostrar resumen
print("\nResumen:")
print(df[['titulo', 'dificultad', 'porciones']].head(10))

Iniciando scraping masivo...
Extrayendo enlaces de las primeras 8 páginas...


100%|██████████| 8/8 [00:20<00:00,  2.57s/it]


Se encontraron 71 URLs únicas de recetas.
Scraping de recetas individuales...


100%|██████████| 71/71 [05:40<00:00,  4.79s/it]



Scraping completado. Se obtuvieron 71 recetas válidas.

Se extrajeron 71 recetas.

✅ Archivos guardados correctamente:
- recetas_extraidas.xlsx
- recetas_extraidas.json

Resumen:
                            titulo dificultad porciones
0      Tarta de manzanas con crema     Normal   Comen 8
1         Bolitas de queso y huevo     Normal   Comen 4
2          Pan de atún sin cocción     Normal   Comen 4
3      Galletas de claras de huevo     Normal   Comen 6
4   Ensalada de verduras grilladas     Normal   Comen 4
5       Botes de berenjenas fritas     Normal   Comen 4
6  Ensalada de patatas a la griega     Normal   Comen 4
7             Gambas a la catalana     Normal   Comen 4
8     Ensalada griega con lentejas     Normal   Comen 4
9               Budín de chocolate     Normal   Comen 8


In [10]:
with open("recetas_extraidas.json", "r", encoding="utf-8") as f:
    recipes = json.load(f)

print(f"Re-limpiando {len(recipes)} recetas...")

for recipe in recipes:
    recipe["ingredientes"] = limpiar_ingredientes(recipe.get("ingredientes", ""))
    recipe["instrucciones"] = limpiar_instrucciones(recipe.get("instrucciones", ""))

with open("recetas_limpias.json", "w", encoding="utf-8") as f:
    json.dump(recipes, f, ensure_ascii=False, indent=2)

df = pd.DataFrame(recipes)
df['ingredientes'] = df['ingredientes'].apply(lambda x: "\n".join(x) if isinstance(x, list) else "")
df['instrucciones'] = df['instrucciones'].apply(lambda x: "\n".join(x) if isinstance(x, list) else "")

df.to_excel("recetas_limpias_v2.xlsx", index=False)

print("✅ Versión 2 generada correctamente")

Re-limpiando 71 recetas...
✅ Versión 2 generada correctamente
